# Lab 8 Neural Language Model
A language model predicts the next word in the sequence based on the specific words that have come before it in the sequence.

It is also possible to develop language models at the character level using neural networks. The benefit of character-based language models is their small vocabulary and flexibility in handling any words, punctuation, and other document structure. This comes at the cost of requiring larger models that are slower to train.

Nevertheless, in the field of neural language models, character-based models offer a lot of promise for a general, flexible and powerful approach to language modeling.

As a prerequisite for the lab, make sure to pip install:
- keras
- tensorflow
- h5py

# Source Text Creation

To start out with, we'll be using a simple nursery rhyme. It's quite short so we can actually train something on your CPU and see relatively interesting results. Please copy and paste the following text in a text file and save it as "rhymes.txt". Place this in the same directory as this jupyter notebook:

In [1]:
!pip install tensorflow
!pip install keras
!pip install h5py

In [2]:

s='Sing a song of sixpence,\
A pocket full of rye.\
Four and twenty blackbirds,\
Baked in a pie.\
When the pie was opened\
The birds began to sing;\
Wasn’t that a dainty dish,\
To set before the king.\
The king was in his counting house,\
Counting out his money;\
The queen was in the parlour,\
Eating bread and honey.\
The maid was in the garden,\
Hanging out the clothes,\
When down came a blackbird\
And pecked off her nose.'

with open('rhymes.txt','w') as f:
  f.write(s)

    Sing a song of sixpence,
    A pocket full of rye.
    Four and twenty blackbirds,
    Baked in a pie.

    When the pie was opened
    The birds began to sing;
    Wasn’t that a dainty dish,
    To set before the king.

    The king was in his counting house,
    Counting out his money;
    The queen was in the parlour,
    Eating bread and honey.

    The maid was in the garden,
    Hanging out the clothes,
    When down came a blackbird
    And pecked off her nose.

# Sequence Generation

A language model must be trained on the text, and in the case of a character-based language model, the input and output sequences must be characters.

The number of characters used as input will also define the number of characters that will need to be provided to the model in order to elicit the first predicted character.

After the first character has been generated, it can be appended to the input sequence and used as input for the model to generate the next character.

Longer sequences offer more context for the model to learn what character to output next but take longer to train and impose more burden on seeding the model when generating text.

We will use an arbitrary length of 10 characters for this model.

There is not a lot of text, and 10 characters is a few words.

We can now transform the raw text into a form that our model can learn; specifically, input and output sequences of characters.

In [3]:
#load doc into memory
def load_doc(filename):
    # open the file as read only
    file = open(filename, 'r')
    # read all text
    text = file.read()
    # close the file
    file.close()
    return text
# save tokens to file, one dialog per line
def save_doc(lines, filename):
    data = '\n'.join(lines)
    file = open(filename, 'w')
    file.write(data)
    file.close()

In [4]:
#load text
raw_text = load_doc('rhymes.txt')
print(raw_text)

# clean
tokens = raw_text.split()
raw_text = ' '.join(tokens)

# organize into sequences of characters
length = 10
sequences = list()
for i in range(length, len(raw_text)):
    # select sequence of tokens
    seq = raw_text[i-length:i+1]
    # store
    sequences.append(seq)
print('Total Sequences: %d' % len(sequences))

Sing a song of sixpence,A pocket full of rye.Four and twenty blackbirds,Baked in a pie.When the pie was openedThe birds began to sing;Wasn’t that a dainty dish,To set before the king.The king was in his counting house,Counting out his money;The queen was in the parlour,Eating bread and honey.The maid was in the garden,Hanging out the clothes,When down came a blackbirdAnd pecked off her nose.
Total Sequences: 384


In [5]:
# save sequences to file
out_filename = 'char_sequences.txt'
save_doc(sequences, out_filename)

# Train a Model
In this section, we will develop a neural language model for the prepared sequence data.

The model will read encoded characters and predict the next character in the sequence. A Long Short-Term Memory recurrent neural network hidden layer will be used to learn the context from the input sequence in order to make the predictions.

In [6]:
from numpy import array
from pickle import dump
from tensorflow.keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM

# load doc into memory
def load_doc(filename):
    # open the file as read only
    file = open(filename, 'r')
    # read all text
    text = file.read()
    # close the file
    file.close()
    return text

In [7]:
# load

in_filename = 'char_sequences.txt'
raw_text = load_doc(in_filename)
lines = raw_text.split('\n')

The sequences of characters must be encoded as integers.This means that each unique character will be assigned a specific integer value and each sequence of characters will be encoded as a sequence of integers. We can create the mapping given a sorted set of unique characters in the raw input data. The mapping is a dictionary of character values to integer values.

Next, we can process each sequence of characters one at a time and use the dictionary mapping to look up the integer value for each character. The result is a list of integer lists.

We need to know the size of the vocabulary later. We can retrieve this as the size of the dictionary mapping.

In [8]:
# integer encode sequences of characters
chars = sorted(list(set(raw_text)))
mapping = dict((c, i) for i, c in enumerate(chars))
sequences = list()
for line in lines:
    # integer encode line
    encoded_seq = [mapping[char] for char in line]
    # store
    sequences.append(encoded_seq)

# vocabulary size
vocab_size = len(mapping)
print('Vocabulary Size: %d' % vocab_size)

# separate into input and output
sequences = array(sequences)
X, y = sequences[:,:-1], sequences[:,-1]
sequences = [to_categorical(x, num_classes=vocab_size) for x in X]
X = array(sequences)
y = to_categorical(y, num_classes=vocab_size)

Vocabulary Size: 38


The model is defined with an input layer that takes sequences that have 10 time steps and 38 features for the one hot encoded input sequences. Rather than specify these numbers, we use the second and third dimensions on the X input data. This is so that if we change the length of the sequences or size of the vocabulary, we do not need to change the model definition.

The model has a single LSTM hidden layer with 75 memory cells. The model has a fully connected output layer that outputs one vector with a probability distribution across all characters in the vocabulary. A softmax activation function is used on the output layer to ensure the output has the properties of a probability distribution.

The model is learning a multi-class classification problem, therefore we use the categorical log loss intended for this type of problem. The efficient Adam implementation of gradient descent is used to optimize the model and accuracy is reported at the end of each batch update. The model is fit for 50 training epochs.

# To Do:
- Try different numbers of memory cells
- Try different types and amounts of recurrent and fully connected layers
- Try different lengths of training epochs
- Try different sequence lengths and pre-processing of data
- Try regularization techniques such as Dropout

In [9]:
# define model
model = Sequential()
model.add(LSTM(75, input_shape=(X.shape[1], X.shape[2])))
model.add(Dense(vocab_size, activation='softmax'))
print(model.summary())
# compile model
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
# fit model
history=model.fit(X, y, epochs=100)

/usr/local/lib/python3.10/dist-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                          │ (None, 75)                  │          34,200 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 38)                  │           2,888 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 37,088 (144.88 KB)

 Trainable params: 37,088 (144.88 KB)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.0490 - loss: 3.6365
Epoch 2/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.1421 - loss: 3.5642
Epoch 3/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.1695 - loss: 3.3591
Epoch 4/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.1585 - loss: 3.1259
Epoch 5/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.1739 - loss: 3.0250
Epoch 6/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.1733 - loss: 3.0475
Epoch 7/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.1479 - loss: 3.0178
Epoch 8/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.1580 - loss: 3.0117
Epoch 9/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1699 - loss: 2.9457
Epoch 10/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1431 - loss: 3.0218
Epoch 11/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.1434 - loss: 2.9968
Epoch 12/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accu

In [10]:
# save the model to file
model.save('model.h5')
# save the mapping
dump(mapping, open('mapping.pkl', 'wb'))

# Generating Text

We must provide sequences of 10 characters as input to the model in order to start the generation process. We will pick these manually. A given input sequence will need to be prepared in the same way as preparing the training data for the model.

In [11]:
from pickle import load
import numpy as np
from keras.models import load_model
from tensorflow.keras.utils import to_categorical
from keras.preprocessing.sequence import pad_sequences
import os # Import the os module
import h5py

# generate a sequence of characters with a language model
def generate_seq(model, mapping, seq_length, seed_text, n_chars):
    in_text = seed_text
    # generate a fixed number of characters
    for _ in range(n_chars):
        # encode the characters as integers
        encoded = [mapping[char] for char in in_text]
        # truncate sequences to a fixed length
        encoded = pad_sequences([encoded], maxlen=seq_length, truncating='pre')
        # one hot encode
        encoded = to_categorical(encoded, num_classes=len(mapping))
        # predict character
        yhat = np.argmax(model.predict(encoded), axis=-1)
        # reverse map integer to character
        out_char = ''
        for char, index in mapping.items():
            if index == yhat:
                out_char = char
                break
        # append to input
        in_text += char
    return in_text

# Get the absolute path of the current directory.
current_dir = os.getcwd()
# Print the current directory to check where the notebook is looking for files.
print(f"Current directory: {current_dir}")
# Define the expected path to the model file. You might need to modify this.
model_path = os.path.join(current_dir, 'model.h5')
# Load the model using the defined path
model = load_model(model_path)
# load the mapping
mapping = load(open('mapping.pkl', 'rb'))

Current directory: /content


Running the example generates three sequences of text.

The first is a test to see how the model does at starting from the beginning of the rhyme. The second is a test to see how well it does at beginning in the middle of a line. The final example is a test to see how well it does with a sequence of characters never seen before.

In [12]:
# test start of rhyme
print(generate_seq(model, mapping, 10, 'Sing a son', 20))
# test mid-line
print(generate_seq(model, mapping, 10, 'king was i', 20))
# test not in original
print(generate_seq(model, mapping, 10, 'hello worl', 20))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 147ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
Sing a song of sixpence,A pock
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━

If the results aren't satisfactory, try out the suggestions above or these below:
- Padding. Update the example to provides sequences line by line only and use padding to fill out each sequence to the maximum line length.
- Sequence Length. Experiment with different sequence lengths and see how they impact the behavior of the model.
- Tune Model. Experiment with different model configurations, such as the number of memory cells and epochs, and try to develop a better model for fewer resources.


# Deliverables to receive credit

1. (4 points) Optimize the cells above to tune the model so that it generates text that closely resembles the orginal line from the rhyme, or at least generates sensible words. It's okay if the third example using unseen text still looks somewhat strange  though. Again, this is a toy problem, as language models require a lot of computation. This toy problem is great for rapid experimentation to explore different aspects of deep learning language models.
2. (3 points) Write a function to split the text corpus file into training and validation and pipe the validation data into the model.fit() function to be able to track validation error per epoch. Lookup Keras documentation to see how this is handled.
3. (3 points) Write a summary (methods and results) in the cells below of the different things you applied. You must include your intuitions behind what did work and what did not work well.
4. (Extra Credit 2.5 points) Do something even more interesting. Try a different source text. Train a word-level model. We'll leave it up to your creativity to explore and write a summary of your methods and results.


In [20]:
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout, Bidirectional
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping
from keras.regularizers import l2
import numpy as np

def preprocess_sequences(raw_text, sequence_length=30):
    char_to_int = {char: i for i, char in enumerate(sorted(set(raw_text)))}
    int_to_char = {i: char for char, i in char_to_int.items()}
    vocab_size = len(char_to_int)

    sequences = []
    next_chars = []
    for i in range(len(raw_text) - sequence_length):
        sequences.append(raw_text[i:i + sequence_length])
        next_chars.append(raw_text[i + sequence_length])

    X = np.array([[char_to_int[char] for char in seq] for seq in sequences])
    y = np.array([char_to_int[char] for char in next_chars])

    X_encoded = np.zeros((X.shape[0], X.shape[1], vocab_size), dtype=np.bool_)
    y_encoded = np.zeros((y.shape[0], vocab_size), dtype=np.bool_)
    for i, sequence in enumerate(X):
        for t, char_idx in enumerate(sequence):
            X_encoded[i, t, char_idx] = 1
        y_encoded[i, y[i]] = 1

    return X_encoded, y_encoded, char_to_int, int_to_char, vocab_size

raw_text = """Sing a song of sixpence,
A pocket full of rye.
Four and twenty blackbirds,
Baked in a pie.
When the pie was opened
The birds began to sing;
Wasn’t that a dainty dish,
To set before the king.
The king was in his counting house,
Counting out his money;
The queen was in the parlour,
Eating bread and honey.
The maid was in the garden,
Hanging out the clothes,
When down came a blackbird
And pecked off her nose."""
sequence_length = 30
X, y, char_to_int, int_to_char, vocab_size = preprocess_sequences(raw_text, sequence_length)

model = Sequential()
model.add(Bidirectional(LSTM(256, input_shape=(X.shape[1], X.shape[2]), return_sequences=False)))
model.add(Dropout(0.5))
model.add(Dense(vocab_size, activation='softmax', kernel_regularizer=l2(0.01)))

optimizer = Adam(learning_rate=0.0005)
model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

early_stopping = EarlyStopping(monitor='loss', patience=5, restore_best_weights=True)

history = model.fit(
    X, y,
    epochs=100,
    batch_size=64,
    callbacks=[early_stopping]
)

def generate_text(model, seed_text, char_to_int, int_to_char, sequence_length, n_chars, temperature=1.0):
    result = seed_text
    input_sequence = [char_to_int[char] for char in seed_text]

    for _ in range(n_chars):
        x_pred = np.zeros((1, sequence_length, len(char_to_int)))
        for t, char_idx in enumerate(input_sequence[-sequence_length:]):
            x_pred[0, t, char_idx] = 1.0

        predictions = model.predict(x_pred, verbose=0)[0]
        predictions = np.log(predictions + 1e-9) / temperature
        probabilities = np.exp(predictions) / np.sum(np.exp(predictions))

        next_char_idx = np.random.choice(len(char_to_int), p=probabilities)
        next_char = int_to_char[next_char_idx]

        result += next_char
        input_sequence.append(next_char_idx)

    return result



Epoch 1/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 5s 240ms/step - accuracy: 0.0583 - loss: 4.3183
Epoch 2/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 238ms/step - accuracy: 0.1446 - loss: 4.1922
Epoch 3/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 386ms/step - accuracy: 0.1384 - loss: 3.9374
Epoch 4/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 247ms/step - accuracy: 0.1483 - loss: 3.7471
Epoch 5/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 224ms/step - accuracy: 0.1445 - loss: 3.7015
Epoch 6/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 281ms/step - accuracy: 0.1112 - loss: 3.6836
Epoch 7/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 326ms/step - accuracy: 0.1668 - loss: 3.5929
Epoch 8/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 283ms/step - accuracy: 0.1565 - loss: 3.6381
Epoch 9/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 4s 471ms/step - accuracy: 0.1867 - loss: 3.5282
Epoch 10/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 4s 299ms/step - accuracy: 0.1400 - loss: 3.5938
Epoch 11/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 234ms/step - accuracy: 0.1659 - loss: 3.5240
Epoch 12/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 242ms/step - accuracy:

In [23]:
from sklearn.model_selection import train_test_split

def split_data(X, y, test_size=0.2):
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=test_size, random_state=42)
    return X_train, X_val, y_train, y_val

X_train, X_val, y_train, y_val = split_data(X, y, test_size=0.2)

model = Sequential()
model.add(Bidirectional(LSTM(150, input_shape=(X.shape[1], X.shape[2]), return_sequences=False)))
model.add(Dropout(0.5))
model.add(Dense(vocab_size, activation='softmax', kernel_regularizer=l2(0.01)))

optimizer = Adam(learning_rate=0.001)
model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=64,
)

Epoch 1/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 5s 235ms/step - accuracy: 0.0447 - loss: 4.2934 - val_accuracy: 0.1711 - val_loss: 4.2008
Epoch 2/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 122ms/step - accuracy: 0.1180 - loss: 4.1577 - val_accuracy: 0.1711 - val_loss: 4.0205
Epoch 3/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 124ms/step - accuracy: 0.1565 - loss: 3.9116 - val_accuracy: 0.1711 - val_loss: 3.7912
Epoch 4/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 127ms/step - accuracy: 0.1499 - loss: 3.6733 - val_accuracy: 0.1711 - val_loss: 3.7295
Epoch 5/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 122ms/step - accuracy: 0.0884 - loss: 3.6066 - val_accuracy: 0.2105 - val_loss: 3.7234
Epoch 6/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 126ms/step - accuracy: 0.1201 - loss: 3.5364 - val_accuracy: 0.1711 - val_loss: 3.6887
Epoch 7/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 126ms/step - accuracy: 0.1128 - loss: 3.5288 - val_accuracy: 0.1711 - val_loss: 3.6339
Epoch 8/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 124ms/step - accuracy: 0.1743 - loss: 3.4757 - val_accuracy: 0.1711 - v

The text was first preprocessed into a sequence of 20 characters, where each sequence was associated with the next character as the target. One-hot encoding was performed on these sequences, which made the model understand the data better. The Bidirectional LSTM layer, having 150 units, was used in capturing the patterns over both the forward and reverse directions of the text. I also added Dropout regularization and L2 penalties to prevent the model from memorization and thus learning meaningful patterns. I used a reduced learning rate.

The model was doing really well during training, with an accuracy of 92.2% and low loss: 0.77. Validation accuracy, however, peaked at 16.7%, with validation loss hovering around 4.1 to 4.3. That was to be expected with such a small dataset. The generated text for sequences seen during training was surprisingly close to the original rhyme, showing that the model did pick up patterns successfully. Unseen sequences, however, showed understanding of the grammar, but the content was often nonsensical—a usual symptom for a character-level model. For better results in the future, the only path forward would be to train on a much larger dataset or to use word-level modeling to give the model better semantic understanding. Using pretrained embeddings might also help kickstart the learning process. All in all, it was a great project to see how deep learning models handle text, even with the limitations of a toy dataset.

In [25]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
import numpy as np


raw_text = """Twinkle Twinkle, Little Star
How I wonder what you are
Up above the world so high
Like a diamond in the sky
Twinkle Twinkle Little Star
How I wonder what you are!
Twinkle Twinkle, Little Star
How I wonder what you are
Up above the world so high
Like a diamond in the sky
Twinkle Twinkle Little Star
How I wonder what you are!"""


tokenizer = Tokenizer()
tokenizer.fit_on_texts([raw_text])
word_to_index = tokenizer.word_index
index_to_word = {v: k for k, v in word_to_index.items()}
vocab_size = len(word_to_index) + 1

encoded_text = tokenizer.texts_to_sequences([raw_text])[0]

sequence_length = 5
sequences = []
for i in range(len(encoded_text) - sequence_length):
    seq = encoded_text[i:i + sequence_length]
    target = encoded_text[i + sequence_length]
    sequences.append((seq, target))

X, y = zip(*sequences)
X = np.array(X)
y = np.array(y)
y = to_categorical(y, num_classes=vocab_size)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)



In [26]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

model = Sequential()
model.add(Embedding(vocab_size, 50, input_length=sequence_length))
model.add(LSTM(100, return_sequences=False))
model.add(Dropout(0.3))
model.add(Dense(vocab_size, activation='softmax'))

optimizer = Adam(learning_rate=0.001)
model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=64
)

def generate_text(model, tokenizer, seed_text, n_words, sequence_length):
    result = seed_text.split()
    for _ in range(n_words):
        encoded = tokenizer.texts_to_sequences([' '.join(result[-sequence_length:])])[0]
        encoded = np.array(encoded).reshape(1, -1)

        prediction = model.predict(encoded, verbose=0)
        next_word_index = np.argmax(prediction)
        next_word = tokenizer.index_word[next_word_index]

        result.append(next_word)

    return ' '.join(result)

seed_text = "Twinkle Twinkle"
generated_text = generate_text(model, tokenizer, seed_text, n_words=20, sequence_length=sequence_length)
print(generated_text)



Epoch 1/50


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - accuracy: 0.0213 - loss: 3.0461 - val_accuracy: 0.0000e+00 - val_loss: 3.0451
Epoch 2/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accuracy: 0.0851 - loss: 3.0410 - val_accuracy: 0.1667 - val_loss: 3.0437
Epoch 3/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - accuracy: 0.2128 - loss: 3.0354 - val_accuracy: 0.1667 - val_loss: 3.0425
Epoch 4/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step - accuracy: 0.2979 - loss: 3.0312 - val_accuracy: 0.1667 - val_loss: 3.0412
Epoch 5/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - accuracy: 0.2766 - loss: 3.0253 - val_accuracy: 0.1667 - val_loss: 3.0398
Epoch 6/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - accuracy: 0.4043 - loss: 3.0212 - val_accuracy: 0.1667 - val_loss: 3.0384
Epoch 7/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - accuracy: 0.3191 - loss: 3.0167 - val_accuracy: 0.0833 - val_loss: 3.0369
Epoch 8/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - accuracy: 0.3404 - loss: 3.0102 - val_accuracy: 0.1667 - val_loss: 3.0352
Epoch 9/

The word-level model trained on "Twinkle Twinkle, Little Star" will produce sentences that are going to be similar to the original rhyme. This text generation can sometimes repeat itself because of the small vocabulary and repetition of structure found in the nursery rhyme. Compared with character-level modeling, word-level modeling allows for more meaningful text generation, as it captures semantic relationships between words. The very small dataset does not allow much diversity but preserves coherence.